In [95]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [96]:
df = pd.read_csv('../week1/week1.csv')
df = df[df['Ticker'] == 'AAPL']
df.describe()

,Open,High,Low,Close,Volume,Target,Close_lag_1,Close_lag_2,Close_lag_3,Daily Return,...,MA_30,STD_7,STD_30,Volume_lag_1,Volume_avg_7,Volume_change,30D RV,MA_ratio,HL_ratio,CO_ratio
count,1828.000000,1828.000000,1828.000000,1828.000000,1.828000e+03,1828.000000,1828.000000,1828.000000,1828.000000,1828.000000,...,1828.000000,1828.000000,1828.000000,1.828000e+03,1.828000e+03,1828.000000,1828.000000,1828.000000,1828.000000,1828.000000
mean,86.281819,87.007898,85.502770,86.276654,8.661293e+07,0.001032,86.199121,86.121275,86.044323,0.102904,...,85.234519,1.315781,2.741445,8.664290e+07,8.672925e+07,0.058650,1.498010,1.010283,1.018289,0.999958
std,33.087843,33.285508,32.882074,33.105044,6.146766e+07,0.015725,33.062492,33.019357,32.977910,1.572628,...,32.544606,0.849648,1.416261,6.145317e+07,5.328477e+07,0.398230,0.475700,0.037990,0.010440,0.012599
min,30.726000,31.070000,30.168000,30.726000,1.124397e+07,-0.123579,30.726000,30.726000,30.726000,-12.357893,...,32.388900,0.081213,0.637318,1.124397e+07,1.722386e+07,-0.708639,0.508498,0.898139,1.004203,0.933734
25%,58.190000,58.684250,57.354000,58.000500,3.929918e+07,-0.006686,57.959500,57.929250,57.882000,-0.669259,...,57.283375,0.698800,1.736409,3.931972e+07,4.155657e+07,-0.196288,1.179467,0.985819,1.011339,0.992695
50%,82.198500,82.764500,81.121000,81.959000,7.228106e+07,0.000885,81.808500,81.707000,81.591000,0.088500,...,80.628400,1.101739,2.424934,7.230770e+07,7.384458e+07,-0.020166,1.417529,1.011298,1.016096,1.000382
75%,109.572500,110.680000,108.640000,109.515000,1.168443e+08,0.009445,109.475000,109.462500,109.445000,0.944476,...,108.733417,1.726495,3.532009,1.168443e+08,1.237581e+08,0.217281,1.742606,1.034381,1.022327,1.007332
max,174.480000,175.610000,173.710000,175.610000,5.192501e+08,0.088725,175.610000,174.180000,173.630000,8.872474,...,160.582000,5.905851,7.947080,5.192501e+08,2.761535e+08,5.054819,3.169211,1.108148,1.182820,1.086940


In [97]:
df.head()

,Ticker,Date,Open,High,Low,Close,Volume,Target,Close_lag_1,Close_lag_2,...,MA_30,STD_7,STD_30,Volume_lag_1,Volume_avg_7,Volume_change,30D RV,MA_ratio,HL_ratio,CO_ratio
0,AAPL,2010-08-10,33.276,33.353,32.984,33.221,125925550,-0.035550,33.520,33.307,...,32.832567,0.155459,0.669400,84312659.0,1.095844e+08,0.493554,1.598099,1.019519,1.011187,0.998347
1,AAPL,2010-08-11,32.676,32.742,31.992,32.040,172348213,0.006429,33.221,33.520,...,32.807033,0.559506,0.684878,125925550.0,1.172171e+08,0.368652,1.513263,1.013816,1.023443,0.980536
2,AAPL,2010-08-12,31.579,32.412,31.521,32.246,148982213,-0.010730,32.040,33.221,...,32.808167,0.656491,0.683887,172348213.0,1.219093e+08,-0.135574,1.481791,1.008129,1.028267,1.021122
3,AAPL,2010-08-13,32.146,32.259,31.898,31.900,98812016,-0.005831,32.246,32.040,...,32.810733,0.724855,0.680196,148982213.0,1.192836e+08,-0.336753,1.478163,1.000313,1.011317,0.992347
4,AAPL,2010-08-16,31.725,32.020,31.582,31.714,88596378,0.017437,31.900,32.246,...,32.813767,0.756828,0.674899,98812016.0,1.204248e+08,-0.103385,1.477501,0.992388,1.013869,0.999653


In [98]:
df['Volume'] = np.log1p(df['Volume'])
df['Volume_avg_7'] = np.log1p(df['Volume_avg_7'])
df['Volume_lag_1'] = np.log1p(df['Volume_lag_1'])
df.columns

Index(['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Target',
       'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Daily Return',
       'Return_lag_1', 'Return_lag_2', 'MA_7', 'MA_30', 'STD_7', 'STD_30',
       'Volume_lag_1', 'Volume_avg_7', 'Volume_change', '30D RV', 'MA_ratio',
       'HL_ratio', 'CO_ratio'],
      dtype='object')

In [99]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.set_index('Date', inplace=True)

def full_decompose(series, model='additive', period=30):
    decomposition = seasonal_decompose(series, model=model, period=period, extrapolate_trend='freq')
    return decomposition.trend, decomposition.seasonal, decomposition.resid

close_trend, close_seasonal, close_resid = full_decompose(df['Close'], model='additive', period=30)
df['Close_trend'] = close_trend
df['Close_seasonal'] = close_seasonal
df['Close_resid'] = close_resid

volume_trend, volume_seasonal, volume_resid = full_decompose(df['Volume'], model='additive', period=30)
df['Volume_trend'] = volume_trend
df['Volume_seasonal'] = volume_seasonal
df['Volume_resid'] = volume_resid
df.dropna(inplace=True)
df.reset_index(inplace=True)

df

,Date,Ticker,Open,High,Low,Close,Volume,Target,Close_lag_1,Close_lag_2,...,30D RV,MA_ratio,HL_ratio,CO_ratio,Close_trend,Close_seasonal,Close_resid,Volume_trend,Volume_seasonal,Volume_resid
0,2010-08-10,AAPL,33.276,33.353,32.984,33.221,18.651201,-0.035550,33.520,33.307,...,1.598099,1.019519,1.011187,0.998347,29.775591,0.055388,3.390020,18.601252,0.037473,0.012477
1,2010-08-11,AAPL,32.676,32.742,31.992,32.040,18.965027,0.006429,33.221,33.520,...,1.513263,1.013816,1.023443,0.980536,29.971357,0.192846,1.875797,18.608702,0.071686,0.284640
2,2010-08-12,AAPL,31.579,32.412,31.521,32.246,18.819337,-0.010730,32.040,33.221,...,1.481791,1.008129,1.028267,1.021122,30.167122,0.111323,1.967555,18.616152,0.048616,0.154569
3,2010-08-13,AAPL,32.146,32.259,31.898,31.900,18.408730,-0.005831,32.246,32.040,...,1.478163,1.000313,1.011317,0.992347,30.362888,-0.122730,1.659842,18.623602,0.004384,-0.219256
4,2010-08-16,AAPL,31.725,32.020,31.582,31.714,18.299602,0.017437,31.900,32.246,...,1.477501,0.992388,1.013869,0.999653,30.558653,-0.046331,1.201678,18.631052,0.010204,-0.341654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1823,2017-11-03,AAPL,173.380,173.640,170.510,171.880,17.903370,0.010182,167.510,166.290,...,1.195754,1.051123,1.018357,0.991348,156.133515,0.189681,15.556804,16.867931,0.041646,0.993794
1824,2017-11-06,AAPL,171.750,174.360,171.100,173.630,17.368033,0.003168,171.880,167.510,...,1.173754,1.061033,1.019053,1.010946,156.107767,0.337548,17.184685,16.860110,0.052934,0.454989
1825,2017-11-07,AAPL,173.290,174.510,173.290,174.180,17.011113,0.008210,173.630,171.880,...,1.151007,1.066748,1.007040,1.005136,156.082019,0.417356,17.680625,16.852290,-0.055917,0.214739
1826,2017-11-08,AAPL,174.030,175.610,173.710,175.610,17.012189,-0.002050,174.180,173.630,...,1.152021,1.070352,1.010938,1.009079,156.056271,0.500669,19.053059,16.844470,-0.021023,0.188742


In [100]:
import yfinance as yf
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

start_date = df.index.min().strftime('%Y-%m-%d')
end_date = df.index.max().strftime('%Y-%m-%d')

symbols = {
    'SP500': '^GSPC',
    'NASDAQ': '^IXIC',
    'VIX': '^VIX'
}

external_data = pd.DataFrame()
for name, symbol in symbols.items():
    temp = yf.download(symbol, start=start_date, end=end_date, progress=False)[['Close']]
    
    if isinstance(temp.columns, pd.MultiIndex):
        temp.columns = temp.columns.get_level_values(0)

    temp = temp.rename(columns={'Close': f'{name}_Close'})
    temp[f'{name}_Return'] = temp[f'{name}_Close'].pct_change()

    external_data = pd.concat([external_data, temp[[f'{name}_Return']]], axis=1)

external_data.index = pd.to_datetime(external_data.index)
df.index = pd.to_datetime(df.index)
df = df.join(external_data, how='left')

df.dropna(inplace=True)
df.reset_index(inplace=True)
import ta
df['ADX'] = ta.trend.adx(df['High'], df['Low'], df['Close'], window=14)

macd = ta.trend.macd(df['Close'], window_slow=26, window_fast=12)
macd_signal = ta.trend.macd_signal(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd
df['MACD_Signal'] = macd_signal
df.dropna(inplace=True)

C:\Users\Akshat\AppData\Local\Temp\ipykernel_3428\1274060054.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  temp = yf.download(symbol, start=start_date, end=end_date, progress=False)[['Close']]
C:\Users\Akshat\AppData\Local\Temp\ipykernel_3428\1274060054.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  temp = yf.download(symbol, start=start_date, end=end_date, progress=False)[['Close']]
C:\Users\Akshat\AppData\Local\Temp\ipykernel_3428\1274060054.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  temp = yf.download(symbol, start=start_date, end=end_date, progress=False)[['Close']]


In [101]:
features = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3',
    'Daily Return', 'Return_lag_1', 'Return_lag_2',
    'MA_7', 'MA_30', 'STD_7', 'STD_30',
    'Volume_lag_1', 'Volume_avg_7', 'Volume_change',
    '30D RV', 'MA_ratio', 'HL_ratio', 'CO_ratio',
    'Close_trend', 'Close_seasonal', 'Close_resid',
    'Volume_trend', 'Volume_seasonal', 'Volume_resid',
    'SP500_Return', 'NASDAQ_Return', 'VIX_Return', 'ADX',
    'MACD', 'MACD_Signal'
]

df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')
df = df.dropna(subset=features + ['Target'])

split = int(0.8 * len(df))
train_df = df.iloc[:split]
test_df = df.iloc[split:]
X_train = train_df[features].copy()
X_test = test_df[features].copy()
y_train = train_df['Target'].copy()
y_test = test_df['Target'].copy()

In [102]:
# import itertools
# from statsmodels.tsa.statespace.sarimax import SARIMAX

# p = d = q = range(0, 3)
# pdq = list(itertools.product(p, d, q))

# best_aic = float("inf")
# best_order = None

# for order in pdq:
#     try:
#         model = SARIMAX(y_train, exog=X_train, order=order, enforce_stationarity=False, enforce_invertibility=False)
#         results = model.fit(disp=False)
#         if results.aic < best_aic:
#             best_aic = results.aic
#             best_order = order
#     except:
#         continue

# print(f"Best ARIMA order found manually: {best_order}")


In [103]:
model = SARIMAX(
    endog=y_train,
    exog=X_train,
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False)
y_pred = results.predict(start=len(y_train), end=len(y_train) + len(y_test) - 1, exog=X_test)
y_pred.index = y_test.index

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
print(f"ARIMAX Model RMSE on Test Set: {rmse:.6f}")
print(f"ARIMAX Model MAE on Test Set: {mae:.6f}")
# Predict on training set
y_train_pred = results.predict(start=0, end=len(y_train) - 1, exog=X_train)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
print(f"ARIMAX Model RMSE on Train Set: {train_rmse:.6f}")
print(f"ARIMAX Model MAE on Train Set: {train_mae:.6f}")

c:\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ARIMAX Model RMSE on Test Set: 0.013508
ARIMAX Model MAE on Test Set: 0.009023
ARIMAX Model RMSE on Train Set: 0.015045
ARIMAX Model MAE on Train Set: 0.011164


c:\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [104]:
X_train.columns

Index(['Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Daily Return',
       'Return_lag_1', 'Return_lag_2', 'MA_7', 'MA_30', 'STD_7', 'STD_30',
       'Volume_lag_1', 'Volume_avg_7', 'Volume_change', '30D RV', 'MA_ratio',
       'HL_ratio', 'CO_ratio', 'Close_trend', 'Close_seasonal', 'Close_resid',
       'Volume_trend', 'Volume_seasonal', 'Volume_resid', 'SP500_Return',
       'NASDAQ_Return', 'VIX_Return', 'ADX', 'MACD', 'MACD_Signal'],
      dtype='object')

In [105]:
close_train_today = df.loc[X_train.index, 'Close'].values
close_test_today = df.loc[X_test.index, 'Close'].values

predicted_close_train_tomorrow = close_train_today * (1 + y_train_pred)
predicted_close_test_tomorrow = close_test_today * (1 + y_pred)

true_close_train_tomorrow = df.loc[y_train.index, 'Close'].shift(-1).values
true_close_test_tomorrow = df.loc[y_test.index, 'Close'].shift(-1).values

predicted_close_train_tomorrow = predicted_close_train_tomorrow[:-1]
true_close_train_tomorrow = true_close_train_tomorrow[:-1]

predicted_close_test_tomorrow = predicted_close_test_tomorrow[:-1]
true_close_test_tomorrow = true_close_test_tomorrow[:-1]

train_price_mae = mean_absolute_error(true_close_train_tomorrow, predicted_close_train_tomorrow)
train_price_rmse = np.sqrt(mean_squared_error(true_close_train_tomorrow, predicted_close_train_tomorrow))

test_price_mae = mean_absolute_error(true_close_test_tomorrow, predicted_close_test_tomorrow)
test_price_rmse = np.sqrt(mean_squared_error(true_close_test_tomorrow, predicted_close_test_tomorrow))

print(f"Close Price Prediction MAE (Train): {train_price_mae:.6f}")
print(f"Close Price Prediction RMSE (Train): {train_price_rmse:.6f}")
print(f"Close Price Prediction MAE (Test): {test_price_mae:.6f}")
print(f"Close Price Prediction RMSE (Test): {test_price_rmse:.6f}")

Close Price Prediction MAE (Train): 0.849461
Close Price Prediction RMSE (Train): 1.187197
Close Price Prediction MAE (Test): 1.191102
Close Price Prediction RMSE (Test): 1.896713


In [106]:
train_direction_acc = (np.sign(y_train_pred) == np.sign(y_train)).mean()
test_direction_acc = (np.sign(y_pred) == np.sign(y_test)).mean()

print("Direction Accuracy (Train):", train_direction_acc)
print("Direction Accuracy (Test):", test_direction_acc)

Direction Accuracy (Train): 0.6373779637377964
Direction Accuracy (Test): 0.6239554317548747


In [107]:
# features += [
#     'Daily Return', '30D RV',
#     'Close_lag_1', 'Close_lag_2', 'Close_lag_3',
#     'Return_lag_1', 'Return_lag_2',
#     'MA_7', 'MA_30', 'STD_7', 'STD_30',
#     'MA_ratio', 'HL_ratio', 'CO_ratio',
#     'Volume_lag_1', 'Volume_avg_7', 'Volume_change',
# ]
X = df[features]
y = df['Target']

split = int(0.8 * len(df))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [108]:
model = RandomForestRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("Train MAE:", train_mae)
print("Train RMSE:", train_rmse)
print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

Train MAE: 0.004490630224629001
Train RMSE: 0.0060907934942416465
Test MAE: 0.00912747168964622
Test RMSE: 0.01242659573717082


In [109]:
train_close_today = df.loc[X_train.index, 'Close'].values
train_predicted_close_tomorrow = train_close_today * (1 + y_train_pred)

train_true_close_tomorrow = df.loc[y_train.index, 'Close'].shift(-1)
train_valid_indices = ~train_true_close_tomorrow.isna()

train_true_close_tomorrow = train_true_close_tomorrow[train_valid_indices].values
train_predicted_close_tomorrow = train_predicted_close_tomorrow[train_valid_indices]

train_mae_close = mean_absolute_error(train_true_close_tomorrow, train_predicted_close_tomorrow)
train_rmse_close = np.sqrt(mean_squared_error(train_true_close_tomorrow, train_predicted_close_tomorrow))

print("Close Price Prediction MAE (Train):", train_mae_close)
print("Close Price Prediction RMSE (Train):", train_rmse_close)

test_close_today = df.loc[X_test.index, 'Close'].values
test_predicted_close_tomorrow = test_close_today * (1 + y_test_pred)
test_true_close_tomorrow = df.loc[y_test.index, 'Close'].shift(-1)
valid_indices = ~test_true_close_tomorrow.isna()

test_true_close_tomorrow = test_true_close_tomorrow[valid_indices].values
test_predicted_close_tomorrow = test_predicted_close_tomorrow[valid_indices]

test_mae_close = mean_absolute_error(test_true_close_tomorrow, test_predicted_close_tomorrow)
test_rmse_close = np.sqrt(mean_squared_error(test_true_close_tomorrow, test_predicted_close_tomorrow))

print("Close Price Prediction MAE (Test):", test_mae_close)
print("Close Price Prediction RMSE (Test):", test_rmse_close)

Close Price Prediction MAE (Train): 0.34216746229766704
Close Price Prediction RMSE (Train): 0.4802032313040898
Close Price Prediction MAE (Test): 1.2041675692885667
Close Price Prediction RMSE (Test): 1.68150973730242


In [110]:
train_direction_true = np.sign(y_train)
train_direction_pred = np.sign(y_train_pred)
train_direction_accuracy = (train_direction_true == train_direction_pred).mean()

test_direction_true = np.sign(y_test)
test_direction_pred = np.sign(y_test_pred)
test_direction_accuracy = (test_direction_true == test_direction_pred).mean()

print("Direction Prediction Accuracy (Train):", train_direction_accuracy)
print("Direction Prediction Accuracy (Test):", test_direction_accuracy)

Direction Prediction Accuracy (Train): 0.9239888423988842
Direction Prediction Accuracy (Test): 0.6072423398328691


In [111]:
# Basic Signal: Buy if predicted close > today's close
test_signals = (predicted_close_test_tomorrow > close_test_today[:-1]).astype(int)  # 1 = Buy, 0 = Sell
# Entry: today's close, Exit: predicted tomorrow's close
entry_prices = close_test_today[:-1]
exit_prices = predicted_close_test_tomorrow

# Real exit prices (ground truth, not predicted)
true_exit_prices = true_close_test_tomorrow

# Stop-loss logic: 5% loss from entry → exit at stop-loss price
stop_loss_pct = 0.05
stop_loss_price = entry_prices * (1 - stop_loss_pct)

# Apply stop-loss: If true price falls below stop-loss, exit at stop-loss
final_exit_prices = np.where(true_exit_prices < stop_loss_price, stop_loss_price, true_exit_prices)

# Calculate returns only when signal == 1 (buy signal)
returns = np.where(test_signals == 1, (final_exit_prices - entry_prices) / entry_prices, 0.0)
# Assume risk-free rate ≈ 0, daily returns
mean_return = np.mean(returns)
std_return = np.std(returns)
sharpe_ratio = (mean_return / std_return) * np.sqrt(252)  # Annualized
print(f"Total Trades: {test_signals.sum()}")
print(f"Average Return per Trade: {mean_return:.4f}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Total Return: {np.sum(returns) * 100:.2f}%")
print(f"Max Drawdown: {(np.min(np.cumsum(returns)) * 100):.2f}%")

Total Trades: 238
Average Return per Trade: 0.0025
Sharpe Ratio: 4.23
Total Return: 90.68%
Max Drawdown: -0.61%
